# IEEE-CID Fraud Detection — Data Preprocessing Pipeline
This notebook covers the full preprocessing pipeline for the IEEE-CID fraud detection dataset, designed for a deep learning (neural network) model.

**Steps covered:**
1. Load & Merge transaction + identity datasets
2. Define column groups
3. Feature engineering (time-based, aggregation, interaction)
4. Clean data (impute, outlier capping)
5. Train / Val / Test split (80 / 5 / 15) with stratification
6. Encode categorical features
7. Scale numerical features
8. Save outputs

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle
import os

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 1. Load & Merge
We do a **left join** on `TransactionID` so that all transactions are kept, even if they have no corresponding identity record (most won't).

In [2]:
transaction = pd.read_csv('data/train_transaction.csv')
identity    = pd.read_csv('data/train_identity.csv')

df = transaction.merge(identity, on='TransactionID', how='left')

print(f'Transaction rows : {len(transaction):,}')
print(f'Identity rows    : {len(identity):,}')
print(f'Merged rows      : {len(df):,}')
print(f'Merged columns   : {df.shape[1]}')

df.head()

Transaction rows : 590,540
Identity rows    : 144,233
Merged rows      : 590,540
Merged columns   : 434


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


## 2. Define Column Groups

In [3]:
target    = 'isFraud'
drop_cols = ['TransactionID']  # ID column, not a feature

# Categorical columns
cat_cols_candidates = [
    'ProductCD',
    'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2',
    'P_emaildomain', 'R_emaildomain',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
    'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29',
    'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38',
    'DeviceType', 'DeviceInfo'
]
cat_cols = [c for c in cat_cols_candidates if c in df.columns]

# Numerical: everything else except target, drops, and categoricals
num_cols = [
    c for c in df.columns
    if c not in ([target] + drop_cols + cat_cols)
]

print(f'Categorical columns : {len(cat_cols)}')
print(f'Numerical columns   : {len(num_cols)}')
print(f'Target              : {target}')

Categorical columns : 37
Numerical columns   : 395
Target              : isFraud


## 3. Feature Engineering
Feature engineering is done **before the split** since frequency-based features need global context.

> ⚠️ **Note:** This means frequency features technically include a small amount of val/test information. This is a common pragmatic trade-off in fraud detection. If you want strict no-leakage, move this section to after the split and compute on train only.

Features added:
- **Time-based**: hour of day, day-of-week proxy, weekend flag
- **Aggregation**: card frequency, per-card transaction amount stats, deviation from card mean
- **Transaction amount**: log transform, cents (decimal part)
- **Interaction**: card+address frequency, card+email frequency
- **M-column summary**: count of T/F/null across M1–M9

In [4]:
# ── Time-based features ──────────────────────────────────────────────────────
df['Transaction_hour']    = (df['TransactionDT'] // 3600) % 24
df['Transaction_day']     = (df['TransactionDT'] // 86400) % 7   # day-of-week proxy
df['Transaction_weekend'] = (df['Transaction_day'] >= 5).astype(int)

# ── Card frequency features ──────────────────────────────────────────────────
for col in ['card1', 'card2', 'card3', 'card4', 'card5', 'card6']:
    if col in df.columns:
        df[f'{col}_freq'] = df[col].map(df[col].value_counts())

# ── Email domain frequency ───────────────────────────────────────────────────
for col in ['P_emaildomain', 'R_emaildomain']:
    if col in df.columns:
        df[f'{col}_freq'] = df[col].map(df[col].value_counts())

# ── Per-card transaction amount stats ───────────────────────────────────────
if 'card1' in df.columns:
    card1_stats = df.groupby('card1')['TransactionAmt'].agg(['mean', 'std', 'count'])
    card1_stats.columns = ['card1_amt_mean', 'card1_amt_std', 'card1_count']
    df = df.merge(card1_stats, on='card1', how='left')
    df['amt_diff_from_card_mean'] = df['TransactionAmt'] - df['card1_amt_mean']
    df['amt_ratio_to_card_mean']  = df['TransactionAmt'] / (df['card1_amt_mean'] + 1e-9)

# ── Transaction amount features ──────────────────────────────────────────────
df['TransactionAmt_log']   = np.log1p(df['TransactionAmt'])
df['TransactionAmt_cents'] = (df['TransactionAmt'] % 1).round(2)

# ── Interaction features ─────────────────────────────────────────────────────
if 'card1' in df.columns and 'addr1' in df.columns:
    combo = df['card1'].astype(str) + '_' + df['addr1'].astype(str)
    df['card1_addr1_freq'] = combo.map(combo.value_counts())

if 'card1' in df.columns and 'P_emaildomain' in df.columns:
    combo = df['card1'].astype(str) + '_' + df['P_emaildomain'].astype(str)
    df['card1_email_freq'] = combo.map(combo.value_counts())

# ── M-column summary features ────────────────────────────────────────────────
m_cols = [c for c in df.columns if c.startswith('M') and c[1:].isdigit()]
if m_cols:
    df['M_true_count']  = (df[m_cols] == 'T').sum(axis=1)
    df['M_false_count'] = (df[m_cols] == 'F').sum(axis=1)
    df['M_null_count']  = df[m_cols].isnull().sum(axis=1)

print(f'Shape after feature engineering: {df.shape}')

C:\Users\65938\AppData\Local\Temp\ipykernel_5576\1501434911.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Transaction_hour']    = (df['TransactionDT'] // 3600) % 24
C:\Users\65938\AppData\Local\Temp\ipykernel_5576\1501434911.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Transaction_day']     = (df['TransactionDT'] // 86400) % 7   # day-of-week proxy
C:\Users\65938\AppData\Local\Temp\ipykernel_5576\1501434911.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

Shape after feature engineering: (590540, 457)


C:\Users\65938\AppData\Local\Temp\ipykernel_5576\1501434911.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['M_null_count']  = df[m_cols].isnull().sum(axis=1)


In [5]:
# Refresh column groups after new features were added
cat_cols = [c for c in cat_cols if c in df.columns]
num_cols = [
    c for c in df.columns
    if c not in ([target] + drop_cols + cat_cols)
]
print(f'Updated — Categorical: {len(cat_cols)}, Numerical: {len(num_cols)}')

Updated — Categorical: 37, Numerical: 418


## 4. Clean Data
- Drop columns with **>90% missing values** (common in this dataset's V-columns)
- **Median impute** numerical NaNs — neural networks cannot handle NaN inputs
- Fill categorical NaNs with a dedicated `'missing'` category
- **Winsorise** numerical outliers at 1st/99th percentile — neural networks are sensitive to extreme values

In [6]:
# ── Drop high-missingness columns ────────────────────────────────────────────
missing_rate  = df[num_cols + cat_cols].isnull().mean()
high_missing  = missing_rate[missing_rate > 0.90].index.tolist()

print(f'Dropping {len(high_missing)} columns with >90% missing values')
df.drop(columns=high_missing, inplace=True)
cat_cols = [c for c in cat_cols if c not in high_missing]
num_cols = [c for c in num_cols if c not in high_missing]

# ── Impute numerical columns ─────────────────────────────────────────────────
for col in num_cols:
    if col in df.columns and df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# ── Impute categorical columns ───────────────────────────────────────────────
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).replace('nan', 'missing')

# ── Winsorise outliers ───────────────────────────────────────────────────────
for col in num_cols:
    if col in df.columns:
        low  = df[col].quantile(0.01)
        high = df[col].quantile(0.99)
        df[col] = df[col].clip(low, high)

# ── Drop ID column ───────────────────────────────────────────────────────────
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

print(f'Shape after cleaning: {df.shape}')
print(f'Remaining nulls: {df.isnull().sum().sum()}')

Dropping 12 columns with >90% missing values
Shape after cleaning: (590540, 444)
Remaining nulls: 10319606


## 5. Train / Val / Test Split (80 / 5 / 15)
We split **before** encoding and scaling to prevent data leakage. Stratification ensures all three splits maintain the same ~3.5% fraud rate.

In [ ]:
# Step 1: split off 15% test
train_val, test = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df[target]
)

# Step 2: split remaining 85% into 80% train and 5% val
# 5% of total = 5/85 ≈ 0.0588 of train_val
train, val = train_test_split(
    train_val, test_size=0.0588, random_state=42, stratify=train_val[target]
)

print(f'Train      : {len(train):,} rows ({len(train)/len(df):.1%}) | fraud rate: {train[target].mean():.3%}')
print(f'Validation : {len(val):,} rows ({len(val)/len(df):.1%})  | fraud rate: {val[target].mean():.3%}')
print(f'Test       : {len(test):,} rows ({len(test)/len(df):.1%}) | fraud rate: {test[target].mean():.3%}')

## 6. Encode Categorical Features
We **label encode** categoricals to integers. For a neural network, these integer codes will typically be passed into `nn.Embedding` layers rather than one-hot encoded — this keeps dimensionality manageable for high-cardinality columns like `card1` or `DeviceInfo`.

Encoders are **fit on train only**, then applied to val and test. Unseen categories in val/test are mapped to an out-of-vocabulary index.

In [ ]:
encoders = {}

for col in cat_cols:
    if col not in train.columns:
        continue

    le = LabelEncoder()
    le.fit(train[col])
    classes = list(le.classes_)

    def safe_transform(series, le=le, classes=classes):
        """Map unseen categories to len(classes) (out-of-vocabulary index)."""
        return series.map(lambda x: le.transform([x])[0] if x in classes else len(classes))

    train[col] = safe_transform(train[col])
    val[col]   = safe_transform(val[col])
    test[col]  = safe_transform(test[col])
    encoders[col] = le

print(f'Encoded {len(encoders)} categorical columns.')

# Vocabulary sizes — useful for defining Embedding layers in your model
vocab_sizes = {col: len(le.classes_) + 1 for col, le in encoders.items()}  # +1 for OOV
print('\nVocabulary sizes (for Embedding layers):')
for col, size in vocab_sizes.items():
    print(f'  {col}: {size}')

## 7. Scale Numerical Features
`StandardScaler` is fit on **train only** and applied to val and test. This is essential for neural networks to converge properly.

In [ ]:
num_cols_to_scale = [c for c in num_cols if c in train.columns and c != target]

scaler = StandardScaler()
train[num_cols_to_scale] = scaler.fit_transform(train[num_cols_to_scale])
val[num_cols_to_scale]   = scaler.transform(val[num_cols_to_scale])
test[num_cols_to_scale]  = scaler.transform(test[num_cols_to_scale])

print(f'Scaled {len(num_cols_to_scale)} numerical columns.')

## 8. Final Check

In [ ]:
print('=== Final Dataset Summary ===')
print(f'Train      : {train.shape}  | nulls: {train.isnull().sum().sum()}')
print(f'Validation : {val.shape}    | nulls: {val.isnull().sum().sum()}')
print(f'Test       : {test.shape}   | nulls: {test.isnull().sum().sum()}')
print(f'\nFraud rate — Train: {train[target].mean():.3%} | Val: {val[target].mean():.3%} | Test: {test[target].mean():.3%}')
print(f'\nNumerical features : {len(num_cols_to_scale)}')
print(f'Categorical features: {len(encoders)}')

train.head()

## 9. Save Outputs

In [ ]:
os.makedirs('preprocessed', exist_ok=True)

train.to_csv('preprocessed/train.csv', index=False)
val.to_csv('preprocessed/val.csv',     index=False)
test.to_csv('preprocessed/test.csv',   index=False)

with open('preprocessed/scaler.pkl',   'wb') as f: pickle.dump(scaler,      f)
with open('preprocessed/encoders.pkl', 'wb') as f: pickle.dump(encoders,    f)

metadata = {
    'cat_cols'    : cat_cols,
    'num_cols'    : num_cols_to_scale,
    'vocab_sizes' : vocab_sizes,
    'target'      : target
}
with open('preprocessed/column_metadata.pkl', 'wb') as f: pickle.dump(metadata, f)

print('Saved to ./preprocessed/')
print('  train.csv, val.csv, test.csv')
print('  scaler.pkl, encoders.pkl, column_metadata.pkl')